In [2]:
import pandas as pd
import re
from pathlib import Path

In [3]:
ROOT_DIR = Path.cwd().parents[0]

RAW_PATH = ROOT_DIR / "data" / "raw" / "hdb_resale_2017_onwards.csv"

df = pd.read_csv(RAW_PATH)

df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [4]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

df.columns

Index(['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range',
       'floor_area_sqm', 'flat_model', 'lease_commence_date',
       'remaining_lease', 'resale_price'],
      dtype='str')

In [5]:

df["month"] = pd.to_datetime(df["month"])


df["floor_area_sqm"] = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")
df["lease_commence_date"] = pd.to_numeric(df["lease_commence_date"], errors="coerce")

In [6]:
def parse_remaining_lease(value):
    """
    Convert remaining lease text into total months.

    Examples:
    '94 years 08 months' -> 1136
    '61 years' -> 732
    """
    if pd.isna(value):
        return None

    value = str(value).lower()

    years = 0
    months = 0

    year_match = re.search(r"(\d+)\s*years?", value)
    month_match = re.search(r"(\d+)\s*months?", value)

    if year_match:
        years = int(year_match.group(1))

    if month_match:
        months = int(month_match.group(1))

    return years * 12 + months


def parse_storey_midpoint(value):
    """
    Convert storey range into midpoint.

    Example:
    '04 TO 06' -> 5
    """
    if pd.isna(value):
        return None

    numbers = re.findall(r"\d+", str(value))

    if len(numbers) >= 2:
        low = int(numbers[0])
        high = int(numbers[1])
        return (low + high) / 2

    return None

In [7]:
df["remaining_lease_months"] = df["remaining_lease"].apply(parse_remaining_lease)

df["remaining_lease_years"] = (
    df["remaining_lease_months"] / 12
)

df["storey_midpoint"] = (
    df["storey_range"].apply(parse_storey_midpoint)
)

df["price_per_sqm"] = (
    df["resale_price"] / df["floor_area_sqm"]
)

df["transaction_year"] = df["month"].dt.year
df["transaction_month"] = df["month"].dt.month

df["flat_age"] = (
    df["transaction_year"] - df["lease_commence_date"]
)

df.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,remaining_lease_months,remaining_lease_years,storey_midpoint,price_per_sqm,transaction_year,transaction_month,flat_age
0,2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,736,61.333333,11.0,5272.727273,2017,1,38
1,2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,727,60.583333,2.0,3731.343284,2017,1,39
2,2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,749,62.416667,2.0,3910.447761,2017,1,37
3,2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,745,62.083333,5.0,3897.058824,2017,1,37
4,2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,749,62.416667,2.0,3955.223881,2017,1,37


In [8]:
df = df.dropna(
    subset=[
        "town",
        "flat_type",
        "floor_area_sqm",
        "resale_price",
        "remaining_lease_years",
        "storey_midpoint",
    ]
)

df.shape

(231870, 18)

In [9]:
PROCESSED_PATH = (
    ROOT_DIR / "data" / "processed" / "hdb_resale_cleaned.csv"
)

PROCESSED_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(PROCESSED_PATH, index=False)



In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 231870 entries, 0 to 231869
Data columns (total 18 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   month                   231870 non-null  datetime64[us]
 1   town                    231870 non-null  str           
 2   flat_type               231870 non-null  str           
 3   block                   231870 non-null  str           
 4   street_name             231870 non-null  str           
 5   storey_range            231870 non-null  str           
 6   floor_area_sqm          231870 non-null  float64       
 7   flat_model              231870 non-null  str           
 8   lease_commence_date     231870 non-null  int64         
 9   remaining_lease         231870 non-null  str           
 10  resale_price            231870 non-null  float64       
 11  remaining_lease_months  231870 non-null  int64         
 12  remaining_lease_years   231870 non-null  

In [11]:
df.describe()

,month,floor_area_sqm,lease_commence_date,resale_price,remaining_lease_months,remaining_lease_years,storey_midpoint,price_per_sqm,transaction_year,transaction_month,flat_age
count,231870,231870.000000,231870.000000,2.318700e+05,231870.000000,231870.000000,231870.000000,231870.000000,231870.000000,231870.000000,231870.000000
mean,2021-11-14 10:42:59.479881,96.707352,1996.526403,5.301639e+05,889.851516,74.154293,8.780528,5542.701069,2021.416846,6.462656,24.890443
min,2017-01-01 00:00:00,31.000000,1966.000000,1.400000e+05,477.000000,39.750000,2.000000,2089.552239,2017.000000,1.000000,1.000000
25%,2019-09-01 00:00:00,81.000000,1985.000000,3.900000e+05,748.000000,62.333333,5.000000,4375.000000,2019.000000,4.000000,10.000000
50%,2021-12-01 00:00:00,93.000000,1997.000000,5.000000e+05,887.000000,73.916667,8.000000,5267.857143,2021.000000,7.000000,25.000000
75%,2024-03-01 00:00:00,112.000000,2012.000000,6.380000e+05,1063.000000,88.583333,11.000000,6298.507463,2024.000000,9.000000,37.000000
max,2026-05-01 00:00:00,366.700000,2022.000000,1.728000e+06,1173.000000,97.750000,50.000000,16148.936170,2026.000000,12.000000,60.000000
std,NaN,24.016303,14.350780,1.896903e+05,171.448104,14.287342,5.950786,1649.411401,2.643751,3.415456,14.256873


In [12]:
df.isna().sum().sort_values(ascending=False)

month                     0
town                      0
flat_type                 0
block                     0
street_name               0
storey_range              0
floor_area_sqm            0
flat_model                0
lease_commence_date       0
remaining_lease           0
resale_price              0
remaining_lease_months    0
remaining_lease_years     0
storey_midpoint           0
price_per_sqm             0
transaction_year          0
transaction_month         0
flat_age                  0
dtype: int64